### TEST ALGO:
1) prep all env files
    - build-env.sh: includes package management
    - uv: pyproject.toml (uv add <lib), uv.lock
    - algorithm-config.yml: specs for registering the algorithm with MAAP API (not GUI)
    - add libs:
        - uv add dask
        - uv sync
        - bash build-env.sh
2) push code to public gitHub repo
3) cd /tmp
4) clone github repo, create outdir and indir
5) cp any data needed to indir
6) bash build-env.sh
7) 

### Submit jobs
follow steps 1 and 2

3) using either GUI or API, register algorithm and allow build
4) Submit jobs!

In [1]:
from maap.maap import MAAP

In [2]:
maap = MAAP(maap_host="api.maap-project.org")

# BEFORE REGISTERING PUSH TO GITHUB

In [34]:
maap.register_algorithm_from_yaml_file("algorithm-config.yml").text

'{"code": 200, "message": {"id": "cc6a3e467bb9f289f30b805b65a5299c53c0782a", "short_id": "cc6a3e46", "created_at": "2026-05-01T15:11:48.000+00:00", "parent_ids": ["f9f5e13db92b1d8777e853588f7e0916ec1eb8bb"], "title": "Registering algorithm: hls-phenometrics", "message": "Registering algorithm: hls-phenometrics", "author_name": "root", "author_email": "root@767a6889dcb3", "authored_date": "2026-05-01T15:11:48.000+00:00", "committer_name": "root", "committer_email": "root@767a6889dcb3", "committed_date": "2026-05-01T15:11:48.000+00:00", "trailers": {}, "extended_trailers": {}, "web_url": "https://repo.maap-project.org/root/register-job-hysds-v4/-/commit/cc6a3e467bb9f289f30b805b65a5299c53c0782a", "stats": {"additions": 0, "deletions": 0, "total": 0}, "status": "created", "project_id": 3, "last_pipeline": {"id": 17875, "iid": 3291, "project_id": 3, "sha": "cc6a3e467bb9f289f30b805b65a5299c53c0782a", "ref": "hysds-v5", "status": "created", "source": "push", "created_at": "2026-05-01T15:11:49

In [10]:
# from maap.maap import MAAP
# maap = MAAP(maap_host="api.maap-project.org")

In [11]:
# https://console.maap-project.org/

In [38]:
# Chesepeake/Temperate urban mixed LULC (18SUJ)
# Madagascar (39KUA)
# Maggie’s Niger tile (31PGR)
# Arctic tile (06WVT) (04WEE)
# Panama (17PLL)
tileID="04WEE"
year=2024
maap.submitJob(identifier=f"{tileID}-{year}-test",
    algo_id="hls-phenometrics",
    version="main",               
    queue="maap-dps-worker-32vcpu-64gb", 
    tile=tileID,
    target_year=year)
#data_dir= "s3://maap-ops-workspace/shared/colinquinn/hls/testing/10day-subset-SERC/",
#output_dir="s3://maap-ops-workspace/shared/colinquinn/HLS_phenometrics/")

{'job_id': '56cc0add-b1e2-49c8-b9d4-a82eb933a34d', 'status': 'success', 'machine_type': None, 'architecture': None, 'machine_memory_size': None, 'directory_size': None, 'operating_system': None, 'job_start_time': None, 'job_end_time': None, 'job_duration_seconds': None, 'cpu_usage': None, 'cache_usage': None, 'mem_usage': None, 'max_mem_usage': None, 'swap_usage': None, 'read_io_stats': None, 'write_io_stats': None, 'sync_io_stats': None, 'async_io_stats': None, 'total_io_stats': None, 'error_details': None, 'response_code': 200, 'outputs': []}

In [31]:
jobs = []
for tile in ["18SUJ", "31PGR"]:#, "39KUA", "04WEE", "17PLL"]:
    for year in list(range(2015, 2025+1)):
        job = maap.submitJob(
            algo_id="hls-phenometrics",
            version="main",
            identifier=f"{tile}-{year}",
            queue="maap-dps-worker-32vcpu-64gb",
            tile=tile,
            target_year=year
        )
        jobs.append(job)

In [33]:
jobs

[{'job_id': '60ee8a22-1bff-4231-a810-bfc0210275a8', 'status': 'success', 'machine_type': None, 'architecture': None, 'machine_memory_size': None, 'directory_size': None, 'operating_system': None, 'job_start_time': None, 'job_end_time': None, 'job_duration_seconds': None, 'cpu_usage': None, 'cache_usage': None, 'mem_usage': None, 'max_mem_usage': None, 'swap_usage': None, 'read_io_stats': None, 'write_io_stats': None, 'sync_io_stats': None, 'async_io_stats': None, 'total_io_stats': None, 'error_details': None, 'response_code': 200, 'outputs': []},
 {'job_id': '1bfb0cbb-67a2-4a1e-917a-b6929a1eae8b', 'status': 'success', 'machine_type': None, 'architecture': None, 'machine_memory_size': None, 'directory_size': None, 'operating_system': None, 'job_start_time': None, 'job_end_time': None, 'job_duration_seconds': None, 'cpu_usage': None, 'cache_usage': None, 'mem_usage': None, 'max_mem_usage': None, 'swap_usage': None, 'read_io_stats': None, 'write_io_stats': None, 'sync_io_stats': None, 'as

In [65]:
r = maap.getJobStatus("7e0a2c08-9002-43fa-9888-4b220812d5b7")
print(r)

Offline


# DELETE ALGO

In [47]:
algorithm_name = "hls-phenometrics"
branch = "v0.1"

# Takes in an algorithm id
maap.deleteAlgorithm(f"{algorithm_name}:{branch}")


<Response [200]>

In [73]:
import psutil
mem = psutil.virtual_memory()
print(f"[MEM] {mem.used/1e9:.1f}GB / {mem.total/1e9:.1f}GB ({mem.percent}%)", flush=True)

[MEM] 57.0GB / 267.3GB (21.3%)


In [74]:
import subprocess
result = subprocess.run(['curl', '-s', 'http://169.254.169.254/latest/meta-data/instance-type'], 
                       capture_output=True, text=True)
print(result.stdout)

r5.8xlarge


In [ ]:
maap.secrets.add_secret("<SECRET_NAME>", "<SECRET_VALUE>")

ex.
response = maap.secrets.add_secret("MY_TOKEN", "98aj48j(774hh9*H")
print(response)
>>> {'secret_name': 'MY_TOKEN'}